# 01-从零理解 LangGraph

### 目录

- 环境安装
- 一、从一个简单问题开始
- 二、LangGraph 是什么？
- 三、State 到底是什么？
- 四、Node 和 Edge
- 五、条件分支
- 六、为什么用 LangGraph？
- 七、动手跑一下
- 八、LangSmith Studio 可视化
- 九、总结

* * *

### 环境安装

先把环境装好。建议 **Python 3.11+**（Studio 要求）。

#### 默认推荐：conda

```bash
conda create -n langgraph python=3.12
conda activate langgraph
pip install langgraph
```

验证安装：

```bash
python -c "from langgraph.version import __version__; print('langgraph', __version__)"
```

#### 备选 1：uv

```bash
mkdir langgraph-demo
cd langgraph-demo
uv init
uv add langgraph
uv run python -c "from langgraph.version import __version__; print('langgraph', __version__)"
```

#### 备选 2：pip + venv

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# Mac/Linux: source .venv/bin/activate
pip install langgraph
python -c "from langgraph.version import __version__; print('langgraph', __version__)"
```

* * *

### 一、从一个简单问题开始

假设你想让 AI 帮你**订机票**。

最简单的做法：写一个函数，调用 AI，拿到结果，结束。


In [1]:
# 最简单的 AI 调用（为了让 notebook 离线也能跑，这里用一个 mock LLM）
def llm(prompt: str) -> str:
    return f"（示例输出）已收到请求：{prompt}"

result = llm("帮我订明天去北京的机票")
print(result)

（示例输出）已收到请求：帮我订明天去北京的机票


这能用。但现实中，订机票可能需要：

-   先查航班（可能要调用搜索工具）
-   如果没票，换日期再查
-   找到票后，问用户确认
-   用户确认后，调用支付接口
-   支付成功，发确认邮件

这就是一个**多步骤、有分支、需要人工参与**的流程。

用普通函数写，代码会变成一团乱麻。**LangGraph 就是来解决这个问题的。**

* * *

### 二、LangGraph 是什么？

一句话：**LangGraph 是一个让你用「流程图」的方式来组织 AI 工作流的工具。**

> **类比：工厂流水线**
> 
> -   **工位**（节点）= 每个工位做一件事（检查、组装、包装）
> -   **传送带**（边）= 连接工位，决定产品去哪
> -   **产品 + 工单**（状态）= 产品在流水线上移动，工单记录当前进度
> -   **流水线布局图**（图）= 整个工厂的组织方式

四个核心概念：

| 概念  | 说明  | 类比  |
| --- | --- | --- |
| State（状态） | 在流水线上传递的「产品 + 工单」。每个工位都能看到它、修改它。 | 外卖订单 |
| Node（节点） | 流水线上的「工位」。每个工位是一个 Python 函数，做一件事。 | 工位  |
| Edge（边） | 连接工位的「传送带」。可以固定路线，也可以根据条件分流。 | 传送带 |
| Graph（图） | 整个「流水线布局图」。把工位和传送带组装起来。 | 布局图 |

* * *

### 三、State 到底是什么？

State 是 LangGraph 中最重要的概念。**它就是一个「字典」**，在所有节点之间传递。

> **类比：外卖订单**
> 
> 想象一个外卖订单在餐厅里流转：
> 
> -   订单上写着：**客人点了什么、地址、备注、当前状态**
> -   接单员看一眼，填上「已接单」
> -   厨师看一眼，做好菜，填上「已出餐」
> -   骑手看一眼，送出去，填上「已送达」
> 
> 这个「订单」就是 State —— **所有人共享的、可以修改的信息表**。

在代码中，State 就是一个 TypedDict（可以理解为「有类型的字典」）：


In [2]:
from typing import TypedDict


class OrderState(TypedDict):
    customer: str  # 客人名字
    dish: str  # 点的菜
    status: str  # 当前状态

每个节点函数都能拿到这个 State，读取里面的信息，然后返回要修改的部分：

In [3]:
def take_order(state: OrderState):
    # 读取：看订单内容
    print(f"客人 {state['customer']} 点了 {state['dish']}")

    # 修改：只返回要改的字段
    return {"status": "已接单"}
    # 其他字段（customer, dish）保持不变

**检查理解：**

初始状态：`{"customer": "小明", "dish": "宫保鸡丁", "status": "新订单"}`

经过「接单员」节点后，状态会变成什么？

-   A. `{"customer": "小明", "dish": "宫保鸡丁", "status": "已接单"}`
-   B. `{"status": "已接单"}`
-   C. `{"customer": "小明", "dish": "宫保鸡丁", "status": "新订单", "note": "已接单"}`

> 答案是 **A**。节点只返回 `{"status": "已接单"}`，其他字段保持不变。LangGraph 会自动合并返回值和原 State。

* * *

### 四、Node 和 Edge：把步骤连起来

**Node（节点）** 就是一个普通的 Python 函数，接收 State，返回要修改的部分。

**Edge（边）** 就是告诉 LangGraph：「这个节点做完后，下一步去哪」。

**最简单的流程：直线**

```text
START → 接单 → 做菜 → 出餐 → END
```

代码怎么写：

In [4]:
from langgraph.graph import StateGraph, START, END

def take_order(state: OrderState):
    # 读取：看订单内容
    print(f"客人 {state['customer']} 点了 {state['dish']}")

    # 修改：只返回要改的字段
    return {"status": "已接单"}
    # 其他字段（customer, dish）保持不变

def cook(state: OrderState):
    print(f"[做菜] 正在做 {state['dish']}")
    return {"status": "已出餐"}


def serve(state: OrderState):
    print(f"[出餐] {state['dish']} 已交给骑手")
    return {"status": "已送达"}


# 1. 创建一个「流水线建造者」
builder = StateGraph(OrderState)

# 2. 添加工位（节点）
builder.add_node("接单", take_order)
builder.add_node("做菜", cook)
builder.add_node("出餐", serve)

# 3. 连接传送带（边）
builder.add_edge(START, "接单")
builder.add_edge("接单", "做菜")
builder.add_edge("做菜", "出餐")
builder.add_edge("出餐", END)

# 4. 编译（把布局图变成可执行的流水线）
graph = builder.compile()

# 5. 运行！
result = graph.invoke(
    {
        "customer": "小明",
        "dish": "宫保鸡丁",
        "status": "新订单",
    }
)

print("节点:", graph.get_graph().nodes)
print("边:", graph.get_graph().edges)
print("\n最终结果:", result)

客人 小明 点了 宫保鸡丁
[做菜] 正在做 宫保鸡丁
[出餐] 宫保鸡丁 已交给骑手
节点: {'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '接单': Node(id='接单', name='接单', data=接单(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '做菜': Node(id='做菜', name='做菜', data=做菜(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '出餐': Node(id='出餐', name='出餐', data=出餐(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}
边: [Edge(source='__start__', target='接单', data=None, conditional=False), Edge(source='做菜', target='出餐', data=None, conditional=False), Edge(source='接单', target='做菜', data=None, conditional=False), Edge(source='出餐', target='__end__', data=None, conditional=False)]

最终结果: {'customer': '小明', 'dish': '宫保鸡丁', 'status': '已送达'}


* * *

### 五、条件分支：让程序做决定

现实中的流程不是直线的。比如：

```text
                ┌─ 有库存 → 正常出餐
接单 → 检查库存 ─┤
                └─ 没库存 → 通知客人换菜
```

这就是**条件边（Conditional Edge）**：根据当前状态，决定走哪条路。


In [5]:
from typing import Literal, TypedDict

from langgraph.graph import StateGraph, START, END


class StockState(TypedDict):
    customer: str
    dish: str
    status: str
    stock: Literal["有库存", "没库存"]


今日菜单 = {"宫保鸡丁", "鱼香肉丝", "番茄炒蛋"}


# 节点函数：只负责“写入状态”，必须返回 dict（增量更新）
def check_stock(state: StockState):
    stock: Literal["有库存", "没库存"] = "有库存" if state["dish"] in 今日菜单 else "没库存"
    return {"stock": stock}


# 路由函数：只负责“选路”，返回分支 key（不是 dict）
def route_by_stock(state: StockState) -> Literal["有库存", "没库存"]:
    return state["stock"]


def fulfill_order(state: StockState):
    print(f"[正常出餐] {state['dish']} 有库存，继续制作")
    return {"status": "已出餐"}


def notify_customer(state: StockState):
    print(f"[通知客人] {state['dish']} 没库存，联系 {state['customer']} 换菜")
    return {"status": "缺货"}


builder_stock = StateGraph(StockState)
builder_stock.add_node("检查库存", check_stock)
builder_stock.add_node("正常出餐", fulfill_order)
builder_stock.add_node("通知客人", notify_customer)

builder_stock.add_edge(START, "检查库存")

# 添加条件边：根据路由函数的返回值走不同路线
builder_stock.add_conditional_edges(
    "检查库存",
    route_by_stock,
    {
        "有库存": "正常出餐",
        "没库存": "通知客人",
    },
)

builder_stock.add_edge("正常出餐", END)
builder_stock.add_edge("通知客人", END)

graph_stock = builder_stock.compile()

print("\n=== case 1: 有库存 ===")
print(
    graph_stock.invoke(
        {"customer": "小明", "dish": "宫保鸡丁", "status": "新订单", "stock": "有库存"}
    )
)

print("\n=== case 2: 没库存 ===")
print(
    graph_stock.invoke(
        {"customer": "小明", "dish": "红烧牛肉面", "status": "新订单", "stock": "有库存"}
    )
)


=== case 1: 有库存 ===
[正常出餐] 宫保鸡丁 有库存，继续制作
{'customer': '小明', 'dish': '宫保鸡丁', 'status': '已出餐', 'stock': '有库存'}

=== case 2: 没库存 ===
[通知客人] 红烧牛肉面 没库存，联系 小明 换菜
{'customer': '小明', 'dish': '红烧牛肉面', 'status': '缺货', 'stock': '没库存'}


* * *

### 六、为什么用 LangGraph？

你可能会想：「我用 if-else 也能写啊，为什么要用 LangGraph？」

| 需求  | 纯 if-else | LangGraph |
| --- | --- | --- |
| 简单流程 | 能写  | 能写  |
| 多步骤 + 分支 | 代码越来越乱 | 图结构清晰 |
| 需要人工确认 | 很难实现 | 内置支持 |
| 记住对话历史 | 自己管理状态 | 自动保存 |
| 并行执行 | 要写多线程 | 声明式 |
| 调试和可视化 | 靠 print | LangSmith Studio |

**简单说：** 当你的 AI 流程超过 3 步、有分支、需要记忆，用 LangGraph 就对了。

* * *

### 七、动手跑一下

下面用前面讲的「外卖订单」例子，完整跑一遍:

In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# ---- 第1步：定义 State（外卖订单）----
class OrderState(TypedDict):
    customer: str  # 客人名字
    dish: str  # 点的菜
    status: str  # 当前状态


# ---- 第2步：定义节点函数 ----
def take_order(state: OrderState):
    print(f"[接单] {state['customer']} 点了 {state['dish']}")
    return {"status": "已接单"}


def cook(state: OrderState):
    print(f"[做菜] 正在做 {state['dish']}")
    return {"status": "已出餐"}


def serve(state: OrderState):
    print(f"[出餐] {state['dish']} 已交给骑手")
    return {"status": "已送达"}


# ---- 第3步：组装图 ----
builder = StateGraph(OrderState)

builder.add_node("接单", take_order)
builder.add_node("做菜", cook)
builder.add_node("出餐", serve)

builder.add_edge(START, "接单")
builder.add_edge("接单", "做菜")
builder.add_edge("做菜", "出餐")
builder.add_edge("出餐", END)

graph = builder.compile()


# ---- 第4步：查看图结构 ----
graph_data = graph.get_graph()
print("节点:", [n.id for n in graph_data.nodes.values()])
print("边:", [(e.source, e.target) for e in graph_data.edges])


# ---- 第5步：运行 ----
result = graph.invoke({
    "customer": "小明",
    "dish": "宫保鸡丁",
    "status": "新订单"
})

print(f"\n最终结果: {result}")

节点: ['__start__', '接单', '做菜', '出餐', '__end__']
边: [('__start__', '接单'), ('做菜', '出餐'), ('接单', '做菜'), ('出餐', '__end__')]
[接单] 小明 点了 宫保鸡丁
[做菜] 正在做 宫保鸡丁
[出餐] 宫保鸡丁 已交给骑手

最终结果: {'customer': '小明', 'dish': '宫保鸡丁', 'status': '已送达'}


运行后你应该看到：

```text
[接单] 小明 点了 宫保鸡丁
[做菜] 正在做 宫保鸡丁
[出餐] 宫保鸡丁 已交给骑手

最终结果: {'customer': '小明', 'dish': '宫保鸡丁', 'status': '已送达'}
```

注意看：`status` 从「新订单」→「已接单」→「已出餐」→「已送达」，每经过一个节点就更新一次。

* * *

### 八、LangSmith Studio 可视化

LangSmith Studio 提供了**免费的可视化界面**，可以实时看到图的执行过程，比 print 直观 100 倍。

**架构说明**

```text
你的电脑                           LangSmith 云端
┌──────────────────┐              ┌──────────────────┐
│  langgraph dev   │              │  smith.langchain  │
│  (本地服务器)     │  ←── 连接 ──→│  .com (前端 UI)   │
│  localhost:2024   │              │                   │
└──────────────────┘              └──────────────────┘
      │                                    │
  执行代码、存储状态                    只是界面，不执行代码
      │                                    │
      ▼                                    ▼
  你的数据在本地                      Studio 连接本地服务器
```

-   **代码执行**：在你电脑上，不上传
-   **Studio UI**：只是一个前端界面，通过 URL 连接本地服务器
-   **API Key 的作用**：把执行轨迹（trace）上传到云端，方便查看历史
-   **没有 API Key**：也能用 Studio，只是 trace 不保存到云端

**1\. 安装 CLI**

```bash
uv add "langgraph-cli[inmem]"
```

> **注意：** 需要 Python 3.11+。如果你是 3.10，先升级：`uv python pin 3.12`

**2\. 创建 langgraph.json**

在项目根目录创建 `langgraph.json`：

```json
{
  "dependencies": ["."],
  "graphs": {
    "demo": "./demo.py:graph"
  },
  "env": "./.env"
}
```

-   `"./demo.py:graph"` = 从 `demo.py` 中找名为 `graph` 的变量
-   `"env": "./.env"` = 读取环境变量配置文件

**3\. 配置 .env**

创建 `.env` 文件（二选一）：

**方法一：纯本地运行（不需要登录）**

```text
LANGSMITH_TRACING=false
```

**方法二：使用 LangSmith（推荐，能看 trace）**

注册 LangSmith 免费账号（`https://smith.langchain.com`），获取 API Key：

```text
LANGSMITH_API_KEY=lsv2_pt_你的key
```

> **注意：** .env 文件里不要加引号，直接写 `KEY=value`。

**4\. 启动 Studio**

```bash
uv run langgraph dev
```

启动后会输出：

```text
🚀 API: http://127.0.0.1:2024
🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
```

浏览器打开 Studio UI 链接。

**界面说明**

| 区域  | 说明  |
| --- | --- |
| 左侧 Graph | 可视化流程图，能看到节点和边 |
| 右侧 Input | 输入测试数据，对应 State 的字段 |
| Submit | 运行图，执行所有节点 |
| Trace | 查看执行轨迹，每步的 State 变化 |

**如何测试**

1.  在右侧 Input 填入：`customer`=小明，`dish`=宫保鸡丁，`status`=新订单
2.  点击 Submit
3.  观察左侧 Graph 中节点高亮执行
4.  点击 Trace 查看每步的 State 变化

* * *

### 九、总结

| 概念  | 一句话解释 | 类比  |
| --- | --- | --- |
| State | 所有节点共享的数据（字典） | 外卖订单 |
| Node | 做一件事的函数 | 流水线工位 |
| Edge | 节点之间的连接 | 传送带 |
| Graph | 把节点和边组装起来 | 工厂布局图 |
| 条件边 | 根据状态决定去哪个节点 | 分流路口 |
| compile() | 编译图，生成可运行对象 | 把图纸变成流水线 |
| langgraph dev | 启动 Studio 可视化调试 | 给流水线装监控 |

**核心记忆：**

**LangGraph = 用流程图的方式组织 AI 工作流**

State 是共享数据，Node 是处理步骤，Edge 是连接关系。

你只需要定义「有哪些步骤」和「步骤之间怎么连」，LangGraph 负责执行。

* * *

> 📖 参考：[LangGraph 官方文档](https://docs.langchain.com/oss/python/langgraph/overview) · [Studio 文档](https://docs.langchain.com/langsmith/quick-start-studio)
